# X5_Q4 — Parámetros como funciones dinámicas (calibración de diseño)

Primer notebook de la **Fase 2 de X5_alt_v2** (ver
[`docs/plans/plan_x5_alt_v2.md`](../docs/plans/plan_x5_alt_v2.md), que continúa
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10).
TODOs en `docs/tracking/todos.md`, sección X5_alt_v2 — **lista estrictamente
secuencial**: cada parámetro se trabaja en 3 pasos consecutivos
(*implementar* → *ejecutar* → *revisar visuales y validar*), no saltar al
siguiente parámetro sin cerrar los 3 del anterior.

Consume la evidencia ya cerrada de `X5_Q1.ipynb`/`X5_Q2.ipynb`/`X5_Q3.ipynb`
(`{valor}_tabla_estado_v2.csv`, `{valor}_q2_5_export_metricas.json`,
`{valor}_q3_4_tabla_final.csv`) para calibrar, por parámetro, una sigmoide
acotada `P̂(s) = piso + (techo-piso)/(1+exp(-pendiente*(s-punto_medio)))`
contra un **target sintético de diseño** (percentil causal del índice
compuesto de 1-2 variables explicativas) — no un target económico de
backtesting. Esa validación económica (parámetro dinámico vs. estático real)
queda para `X5_Q5.ipynb`, fuera de alcance de este notebook.

**No implementa reglas de trading ni código de producción** — mismo criterio
de alcance que `Q3-3` aplicó a `X5_Q3.ipynb`.

Gate de arranque (`[Q4-1]`): requiere `Q3-EXEC` ya satisfecho — ver
`docs/tracking/done.md`.

Mismo input por lista `valores` que `X5_Q1`/`X5_Q2`/`X5_Q3.ipynb`.

In [ ]:
import sys
from pathlib import Path

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402

valores = list(cfg.VALORES)  # default = todos los activos de config.py; recortar a mano acá para un análisis puntual con menos

## 0. Preparación (Q4-1)

Imports y configuración común. Igual patrón que `X5_Q1`/`X5_Q2`/`X5_Q3.ipynb`:
`config.py` para rutas de `resources/x5_alt/`, opciones de despliegue de
`pandas`.

In [ ]:
import json

import pandas as pd

from x5_alt_funciones import calibrar_parametro

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

print(f"Activos: {valores}")

## 1. Carga de outputs de Q1/Q2/Q3 (Q4-1)

Lee, por cada activo de `valores`, `{valor}_tabla_estado_v2.csv` (`X5_Q1.ipynb`),
`{valor}_q2_5_export_metricas.json` (`X5_Q2.ipynb`) y `{valor}_q3_4_tabla_final.csv`
(`X5_Q3.ipynb`) — ninguno se recalcula acá. Si falta alguno, corrió incompleto
el notebook correspondiente.

In [ ]:
def _cargar_tabla_estado(valor):
    path = cfg.CARPETA_X5_ALT / f'{valor}_tabla_estado_v2.csv'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q1.ipynb primero para generarla.")
    return pd.read_csv(path, parse_dates=['DateTime'])


def _cargar_export_metricas(valor):
    path = cfg.CARPETA_X5_ALT / f'{valor}_q2_5_export_metricas.json'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q2.ipynb primero para generarlo.")
    with open(path, encoding='utf-8') as f:
        return json.load(f)


def _cargar_tabla_final_q3(valor):
    path = cfg.CARPETA_X5_ALT / f'{valor}_q3_4_tabla_final.csv'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q3.ipynb primero para generarla.")
    return pd.read_csv(path)


tablas = {valor: _cargar_tabla_estado(valor) for valor in valores}
export_metricas = {valor: _cargar_export_metricas(valor) for valor in valores}
tablas_finales_q3 = {valor: _cargar_tabla_final_q3(valor) for valor in valores}

for valor in valores:
    tabla = tablas[valor]
    print(f"{valor}: tabla_estado {tabla.shape[0]} filas x {tabla.shape[1]} columnas — "
          f"{tabla['DateTime'].min()} → {tabla['DateTime'].max()}")

## 2. Alcance de este notebook (Q4-2)

**`X5_Q4.ipynb` no implementa reglas de trading ni resultado económico.**
Calibra, por parámetro, una función de diseño `P̂(s)` acotada entre un piso y
un techo configurados (no necesariamente el rango completo de
`X5_PARAM_RANGES` — ver `plan_x5_alt_v2.md` sección 5), ajustada contra un
**target sintético por percentil causal** — no contra un óptimo económico
real (eso es `X5_Q5.ipynb`).

Quedan explícitamente fuera de este notebook:

- Cualquier wiring a `X0_data_supports.py`/`X1_trading.py`/`config.py` de
  producción.
- Separar `A` en `A_OE`/`A_TRAILING` — decisión pendiente de
  `x5_contexto_conversacion.md` §21; `A` se calibra único por ahora.
- `LAMBDA` — fuera del alcance comprometido de esta fase (decisión explícita
  de Mauricio). Queda como extensión opcional al final del notebook
  (Sección 11, `[Q4-L1]`-`[Q4-L4]`), condicional a resolver antes su
  bloqueador de rango (heredado de `Q3-6`).
- Validar si el parámetro dinámico mejora el resultado económico vs. el
  estático actual — eso es `X5_Q5.ipynb`.

Un buen ajuste acá (R² alto contra el target sintético) **no** es evidencia
de utilidad económica — es solo evidencia de que la sigmoide aproxima bien
el mapeo por percentiles que se le pidió aproximar.

## 3. Utilidades compartidas (Q4-2, Q4-3)

`scripts/x5_alt_funciones.py` — sigmoide acotada, índice compuesto (1-2
variables z-scoreadas), percentil causal sin leakage (`[Q4-3]`) y
`calibrar_parametro` (ajuste por `curve_fit` contra el target sintético; con
2 variables explora además el peso relativo por grid search de R², ver
docstring del módulo). Una sola implementación — las secciones 4-9 la
reutilizan, nunca reimplementan la fórmula. Importado ya en la Sección 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from x5_alt_funciones import percentil_causal

VARIABLE_INSPECCION_Q4_3 = 'x3_drawdown_20'  # inspección puntual de Q4-3; no es la variable final de ningún parámetro

for valor in valores:
    tabla = tablas[valor]
    q_t = percentil_causal(tabla[VARIABLE_INSPECCION_Q4_3].to_numpy())

    fig, ax1 = plt.subplots(figsize=(14, 4))
    ax1.plot(tabla['DateTime'], tabla['Close'], color='#1f77b4', linewidth=0.8)
    ax1.set_ylabel('Close', color='#1f77b4')
    ax1.tick_params(axis='y', labelcolor='#1f77b4')

    ax2 = ax1.twinx()
    ax2.plot(tabla['DateTime'], q_t, color='#d62728', linewidth=0.8, alpha=0.85)
    ax2.set_ylabel(f'percentil_causal({VARIABLE_INSPECCION_Q4_3})', color='#d62728')
    ax2.tick_params(axis='y', labelcolor='#d62728')
    ax2.set_ylim(0, 1)

    ax1.set_xlabel('DateTime')
    ax1.set_title(f"{valor} — Close (izq.) vs. percentil_causal({VARIABLE_INSPECCION_Q4_3}) (der.)")
    plt.tight_layout()
    plt.show()

    print(f"{valor}: percentil_causal({VARIABLE_INSPECCION_Q4_3}) — "
          f"media {q_t.mean():.3f}, p05 {np.quantile(q_t, 0.05):.3f}, "
          f"p50 {np.quantile(q_t, 0.5):.3f}, p95 {np.quantile(q_t, 0.95):.3f}")


## 4. N = f(drawdown, volatilidad)

**Pasos**: `[Q4-4]` implementar → `[Q4-5]` ejecutar `X5_Q4.ipynb` completo →
`[Q4-6]` revisar visuales y validar con Mauricio antes de pasar a `A`.

*Pendiente de implementar.* Variables: `x3_drawdown_20` (mejor evidencia Q3
entre las 3 candidatas de drawdown) + `x3_atr_pct` (volatilidad — no estaba en
la tabla `Q3-4` para `N`, se agrega para completar la hipótesis de
`X5_alternativo.md` §10.5). Rango: `(50,200)` BTCUSD/ETHUSD, `(40,180)` resto
(`X5_PARAM_RANGES['n_sizes_ejecucion']`).

Por cada activo: índice compuesto (gráfico de cada variable normalizada + el
índice combinado, a doble eje con `Close`), ajuste con `calibrar_parametro`,
curva ajustada sobre el scatter `(s, y_objetivo)`, serie temporal `N̂_t` vs.
el valor estático actual (180) a doble eje con el precio, diagnóstico (R²,
elasticidad, signo vs. hipótesis Q3), export a `{valor}_q4_n.csv`.

## 5. A = f(volatilidad)

**Pasos**: `[Q4-7]` implementar → `[Q4-8]` ejecutar → `[Q4-9]` revisar
visuales y validar antes de pasar a `factor_B`.

*Pendiente de implementar.* Variable: `x3_atr_pct` (se descarta `x3_bb_width`
por redundante con `atr_pct`, Pearson 0.74, `Q2-12`). Rango:
`(2.0, 20.0)` USD (`X5_PARAM_RANGES['A']`).

**Nota de alcance**: se calibra `A` único, sin separar `A_OE`/`A_TRAILING`
(decisión pendiente, `x5_contexto_conversacion.md` §21) — señalar
explícitamente este supuesto en el notebook al reportar el resultado.

Mismo esqueleto de gráficos/diagnóstico/export que la Sección 4, adaptado a
`k=1`.

## 6. factor_B = f(volatilidad, tendencia)

**Pasos**: `[Q4-10]` implementar → `[Q4-11]` ejecutar → `[Q4-12]` revisar
visuales y validar antes de pasar a `K`.

*Pendiente de implementar.* Variables: `x3_atr_pct` (volatilidad) + `adx_14`
(tendencia — se descarta `regimen_tendencia` por ser nominal, no apto para
un índice continuo). Rango tentativo: `(0.2, 0.7)` (`x5_contexto_conversacion.md`
§8 — **no está en `X5_PARAM_RANGES`** todavía, ver `plan_x5_alt_v2.md`
sección 6).

Mismo esqueleto de gráficos/diagnóstico/export que la Sección 4, adaptado a
`k=2`.

## 7. K = f(persistencia de niveles)

**Pasos**: `[Q4-13]` implementar → `[Q4-14]` ejecutar → `[Q4-15]` revisar
visuales y validar antes de pasar a `N_EXP`.

*Pendiente de implementar.* Variable: `ruptura_rango` (se descarta
`duracion_en_rango`, evidencia ≈0 en `Q3-4`). Rango: `(0.5, 2.0)`
(`X5_PARAM_RANGES['K']`).

Nota: `K` es global en producción (no por activo, `pars_configurables.md`) —
se calibra por activo igual que el resto por consistencia con `Q1`-`Q3`, pero
llevarlo a producción exigiría antes decidir si `K` pasa a ser por-activo
(fuera de alcance, ver `plan_x5_alt_v2.md` sección 6).

## 8. N_EXP = f(cambio de volatilidad)

**Pasos**: `[Q4-16]` implementar → `[Q4-17]` ejecutar → `[Q4-18]` revisar
visuales y validar antes de pasar al resumen consolidado.

*Pendiente de implementar.* Variable: `cambio_volatilidad` (ratio volatilidad
corta/larga — se descarta `frecuencia_extremos_30d` para mantener `k=1`).
Rango: `(0.5, 3.0)` (`X5_PARAM_RANGES['N_EXP']`).

Misma nota que la Sección 7 sobre `K`/`N_EXP`/`LAMBDA` siendo globales en
producción.

## 9. Resumen consolidado (de los 5 parámetros: N, A, factor_B, K, N_EXP)

**Pasos**: `[Q4-19]` implementar → `[Q4-20]` ejecutar y revisar la tabla
completa.

*Pendiente.* Tabla única por activo con bondad de ajuste (R²), elasticidad
en el punto medio y señal de coincidencia de signo vs. la hipótesis de `Q3`,
para los 5 combos calibrados. Export a `{valor}_q4_resumen.csv`.

## 10. Export a zip (Q4-EXEC)

*Pendiente.* `exportar_zip('X5_Q4', archivos)` (`scripts/x5_alt_export.py`)
sobre los CSV generados en las secciones 4-9, mismo patrón que
`X5_Q1`/`X5_Q2`/`X5_Q3.ipynb`. Última corrida de punta a punta (6 activos) —
cierre formal de la Fase 2 principal (sin `LAMBDA`, ver Sección 11).

## 11. Extensión opcional — LAMBDA (fuera del alcance comprometido de esta fase)

Mauricio decidió dejar `LAMBDA` explícitamente fuera del alcance de esta
primera iteración (`plan_x5_alt_v2.md` sección 0). Retomar esta sección solo
si decide resolver su bloqueador — no es parte del cierre de `[Q4-EXEC]`.

**Pasos**: `[Q4-L1]` resolver y documentar el rango real de `LAMBDA` (el
rango de exploración documentado, `0.001`-`0.02`, no coincide con el valor
productivo `0.2`, `config.py:95` — bloqueador heredado de `Q3-6`) →
`[Q4-L2]` implementar `LAMBDA = f(x3_bb_width, duracion_en_rango)` (`k=2`,
mismo esqueleto de las secciones 4-8) → `[Q4-L3]` ejecutar `X5_Q4.ipynb`
completo → `[Q4-L4]` revisar visuales y validar.